[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/condensate/condensate.ipynb)

# A Rotating Bose–Einstein Condensate

This example computes the ground state of a rotating Bose–Einstein condensate in a harmonic trap, where quantized vortices arrange into a triangular lattice. A condensate shares one quantum state across all of its atoms, so its superfluid velocity follows the spatial turning of that wave rather than rigid-body rotation. Circulation concentrates at quantized vortices, density holes around which the wave turns once, and under steady rotation the cloud admits vortices at Feynman's areal density of twice the rotation rate over the circulation quantum, as in rotating superfluid helium.

In the product of the trap plane and an internal phase plane, the condensate at each vertex is modelled as an internal vector whose norm is the square root of the atom density and whose direction sets the local phase. Two bivectors govern the rotating frame: `xy` turns positions in the trap plane by a quarter turn via `point.commutator(xy)`, while `ab` turns the wave by a quarter turn in its internal plane via `wave * ab` and commutes with every spatial blade. The term ***simplex gradient*** here means the corner slope `(wave_plane & sides) / double_areas`, which pairs with the rotational drag velocity `gradients & drag_velocities` against `wave * ab` to couple the two planes directly, while summing the edge rotor logarithms `(ends[0] * ends[1]).normalized().log()` around each triangle boundary yields integer vortex windings without angle unwrapping.

In the complex scalar notation of the Gross–Pitaevskii equation ([Fetter, 2009](https://arxiv.org/abs/0801.2952)), a single imaginary unit $i$ represents both the internal phase turn and the factor multiplying the differential angular-momentum operator $L_z = -i(x\partial_y - y\partial_x)$ inside $E[\psi] = \int \bigl(\tfrac12|\nabla\psi|^2 + V(r)|\psi|^2 + \tfrac g2|\psi|^4 - \Omega\,\psi^*L_z\psi\bigr)\,dx\,dy$, and discrete plaquette winding numbers require branch-cut wrapping $\operatorname{atan2}(\operatorname{Im}(\psi_j^*\psi_k), \operatorname{Re}(\psi_j^*\psi_k))$ around each triangle.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from examples.quantum.condensate import render, scenarios
from examples.util.animation import show_states
from numga import Algebra, NumpyContext
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology

# The trap's plane joined with the condensate's internal phase plane.
ga = Algebra("x+y+") * Algebra("a+b+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Plane = ga.gatype.vector()                                                    # Plane+
Point = ga.gatype.antivector()                                                # Point-
Wave = Plane.derive.within(Algebra("a+b+"))                                   # Wave+
Turn = ga.gatype.bivector().derive.within(Algebra("a+b+"))                    # Turn+

# Unit displacements across and up the trap plane from its centre.
horizontal = mv.x.dual()                                                      # [] Point-
vertical = mv.y.dual()                                                        # [] Point-
# The condensate's internal plane as an intrinsic patch orthogonal to spatial displacements.
wave_plane = mv.a.dual() & mv.b.dual()                                        # [] Antibivector-
# Unit inverse-mass duality map turning a spatial gradient plane into the flow it drives.
mobility = Plane.dual()                                                       # [] Point- <- Plane+

## Imaginary-time relaxation into the vortex lattice

On the concentric triangulated disk of the harmonic trap, the condensate starts as a broad Gaussian cloud stirred by small amplitude and phase noise so rotational symmetry can break cleanly. Joining each opposite triangle edge with `wave_plane` forms the simplex gradients purely in the spatial directions, and pairing them across the inverse-mass duality map `Plane.dual()` gives the kinetic coupling on each triangle.

Each imaginary-time step solves the kinetic, harmonic-trap, and rotational-drag terms implicitly with a single pre-factored vertex operator while advancing contact repulsion explicitly and rescaling to unit atom number. This relaxation and animation cell runs in roughly ten seconds.

In [ ]:
topology, coordinates = scenarios.trap_disk(scenarios.n_rings)
vertices = (horizontal * coordinates[:, 0] + vertical * coordinates[:, 1]).field()   # Point-[V]
trap_potential = mv.scalar((0.5 * (coordinates ** 2).sum(axis=-1))[:, None]).field()   # Scalar+[V]
amplitudes, angles = scenarios.initial_cloud(coordinates)                     # [V], [V]
phase_rotors = (mv.ab * (-angles / 2)).exp()                                  # [V] Even+
initial_wave = (amplitudes * (phase_rotors >> mv.a)).field()                  # Wave+[V]

In [ ]:
def corner_assembly(topology: Topology, local: Wave) -> SparseExtensor:
    """Each triangle's `[3, 3] (Wave+ <- Wave+)[F]` corner coupling pulls back across vertex-triangle incidence to a
    global `(Wave+ <- Wave+)[V <- V]` operator."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    return (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)


def triangle_geometry(topology: Topology, vertices: Point) -> tuple[Scalar, Plane, Scalar]:
    """Joining edge displacements with `wave_plane` determines each triangle's area (`Scalar+[F]`), its three corner
    simplex gradients (`[3] Plane+[F]`), and each vertex's lumped dual area (`Scalar+[V]`)."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [3] Point-[F]
    # Joining two edge displacements with the internal wave plane gives twice the signed triangle area.
    double_areas = (sides[0] & sides[1]) & wave_plane                         # Antiscalar-[F]
    areas = double_areas.antinorm() / 2                                       # Scalar+[F]
    # Joining the internal wave plane with each opposite edge and dividing by twice the area gives the simplex gradient.
    gradients = (wave_plane & sides) / double_areas                           # [3] Plane+[F]
    dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)               # Scalar+[V]
    return areas, gradients, dual_areas


areas, gradients, dual_areas = triangle_geometry(topology, vertices)

In [ ]:
def relaxation_states(topology: Topology, vertices: Point, areas: Scalar, gradients: Plane, dual_areas: Scalar,
                      trap_potential: Scalar, repulsion: float, rotation: float, wave: Wave, step: float,
                      n_steps: int, stride: int) -> Iterator[Wave]:
    """Yields the condensate wave (`Wave+[V]`) every `stride` steps of imaginary-time relaxation at unit atom number,
    stepping kinetic, trap, and rotational-drag terms implicitly and contact repulsion explicitly."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    # Pairwise coupling of corner simplex gradients through the inverse-mass mobility over each triangle.
    kinetic = (gradients[:, None] & mobility(gradients[None, :])) * (areas / 2)   # [3, 3] Scalar+[F]
    # Quarter-turn velocity of each triangle centroid in the rotating frame, paired extrinsic-first with the gradients.
    drag_velocities = (corners.mean(axis=0) * vertices).commutator(mv.xy)     # Point-[F]
    advection = (areas / 3) * (gradients & drag_velocities)[None, :]          # [1, 3] Scalar+[F]
    local_operator = kinetic * Wave - rotation * advection * (Wave * mv.ab)   # [3, 3] (Wave+ <- Wave+)[F]
    step_operator = ((dual_areas / step + dual_areas * trap_potential) * Wave +
                     corner_assembly(topology, local_operator))               # (Wave+ <- Wave+)[V <- V]
    for index in range(n_steps + 1):
        if index % stride == 0:
            yield wave
        density = wave.scalar_norm_squared()                                  # Scalar+[V]
        unnormalized = step_operator.solve(dual_areas * (wave / step - repulsion * density * wave))   # Wave+[V]
        atom_count = (dual_areas * unnormalized.scalar_norm_squared()).sites.sum()                    # [] Scalar+
        wave = unnormalized / atom_count.square_root()                        # Wave+[V]


states = relaxation_states(
    topology, vertices, areas, gradients, dual_areas, trap_potential,
    scenarios.repulsion, scenarios.rotation, initial_wave, scenarios.step, scenarios.n_steps, scenarios.stride,
)
drawn_states = show_states(
    islice(states, scenarios.n_drawn_frames),
    render.wave_animation, topology, vertices,
    name="condensate", duration_ms=scenarios.frame_duration_ms,
    n_frames=scenarios.n_drawn_frames, colors=scenarios.n_animation_colours,
)
settled_states = drawn_states + list(states)

Hue is the phase of the wave, which runs once through the colour wheel around each dark vortex core, and brightness is the atom density against the cloud around it, fading to black at the cores and at the cloud's rim. Quantized vortices nucleate along the dilute rim, migrate inward, and lock into a near-equilateral triangular lattice over the first half of the relaxation, while the remaining steps settle the lattice positions.

## Quantized vortex windings and Feynman's rule

Along every edge of the triangulated disk, the normalized product of the endpoint wave vectors forms the rotor between them, and its bivector logarithm records the signed turn in the internal phase plane. Summing those edge turns around each triangle boundary identifies every vortex core by an integer winding of one full turn, and counting the vortices enclosed within concentric circles tests Feynman's areal density law across the cloud.

In [ ]:
def triangle_windings(topology: Topology, wave: Wave) -> Turn:
    """Summing the logarithms of normalized edge rotors around each triangle boundary gives its phase winding
    (`Turn+[F]`) in whole turns."""
    ends = topology.incidence[1, 0] * wave                                    # [2] Wave+[E]
    edge_turns = (ends[0] * ends[1]).normalized().log()                       # Turn+[E]
    return (topology.boundary[2, 1] * edge_turns).sum(axis=0) / (2 * np.pi)   # Turn+[F]


def vortex_counts(topology: Topology, vertices: Point, wave: Wave, radii: np.ndarray,
                  vortex_threshold: float) -> np.ndarray:
    """Triangles whose phase winding exceeds `vortex_threshold` accumulate by centroid distance into enclosed vortex
    counts (`[n_radii]`) across `radii` (`[n_radii]`)."""
    centroids = topology.incidence[2, 0].mean(axis=0) * vertices              # Point-[F]
    turns = triangle_windings(topology, wave).scalar_product(~mv.ab)          # Scalar+[F]
    vortices = (turns > vortex_threshold) | (turns < -vortex_threshold)       # [F]
    return ((centroids.antinorm().batch() <= radii[:, None]) & vortices).sum(axis=-1)   # [n_radii]


settled_wave = settled_states[-1]                                             # Wave+[V]
n_radii = scenarios.n_radii
radii = scenarios.counting_radii                                              # [n_radii]
counts = vortex_counts(topology, vertices, settled_wave, radii, scenarios.vortex_threshold)   # [n_radii]
feynman_counts = scenarios.rotation * radii ** 2                              # [n_radii]
render.count_figure(radii, counts, feynman_counts);

The blue staircase shows the quantized vortices enclosed within each radius, and the orange curve shows Feynman's uniform-vorticity prediction of the rotation rate times the squared radius. The enclosed count tracks Feynman's parabola across the dense interior of the cloud before levelling off near the Thomas–Fermi rim where the atom density vanishes.

In [ ]:
# checks
# Unit atom number holds on the settled condensate wave.
atom_count = (dual_areas * settled_wave.scalar_norm_squared()).sites.sum()    # [] Scalar+
assert (atom_count - 1).abs() < 1e-10

# Every vortex triangle winds once in the sense set by the rotation, and every other triangle has zero winding.
turns = triangle_windings(topology, settled_wave).scalar_product(~mv.ab)      # Scalar+[F]
vortices = (turns > scenarios.vortex_threshold) | (turns < -scenarios.vortex_threshold)   # [F]
assert ((turns + 1).abs() < 1e-6)[vortices].all()
assert (turns.abs() < 1e-6)[~vortices].all()

# Across the dense interior of the cloud, the enclosed vortex count matches Feynman's areal law within 20%.
bulk = radii == 4.0                                                           # [n_radii]
assert np.allclose(counts[bulk], feynman_counts[bulk], rtol=0.2)